In [ ]:

!pip install -U spacy
!python -m spacy download ru_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 60.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [1]:
import numpy as np
import string
import re
import spacy

translator = str.maketrans('', '', string.punctuation + '«»“”‘’')
nlp = spacy.load("ru_core_news_sm")

file = open('sholohov-don.txt', 'r', encoding='utf-8')

text = file.read()
text = re.sub(r'\d+', '', text)
text_changed = text.translate(translator).lower().split()[1000:4900]
print(len(text_changed))
text_changed = [nlp(word)[0].lemma_ for word in text_changed if len(word) >= 3]
dictionary = {}
print(text_changed[:100])
print(len(text_changed))
print(text_changed[:500])

class Word2Vec:
    def __init__(self, words, L_size, vector_size):
        self.L_size = L_size
        self.vector_size = vector_size
        dictionary_words = {}
        dictionary_indexes = {}
        for i in range(len(list(words))):
            dictionary_words[words[i]] = i
            dictionary_indexes[i] = words[i]
        self.dictionary_words = dictionary_words
        self.dictionary_indexes = dictionary_indexes
        self.dictionary_size = len(list(words))
        self.W = np.random.rand(self.dictionary_size, self.vector_size)
        self.C = np.random.rand(self.vector_size, self.dictionary_size)

        dictionary_context = {}
        for i in range(len(words)):
            context = []
            for j in range(max(0, i - int(self.L_size / 2)), min(len(words), i + int(self.L_size / 2) + 1)):
                if (i != j):
                    context.append(self.dictionary_words[words[j]])
            dictionary_context[tuple(set(context))] = self.dictionary_words[words[i]]
        self.dictionary_context = dictionary_context

    def cross_entropy_loss(self, prediction, target):
        return -np.sum(target * np.log(prediction + 1e-10))

    def softmax(self, vector):
        exponent = np.exp(vector - np.max(vector))
        return exponent / np.sum(exponent)

    def train(self, words, epochs, learning_rate):
        for epoch in range(epochs):
            loss = 0
            for context, target in self.dictionary_context.items():
                context_vector = np.zeros((1, self.dictionary_size))
                for k in range(len(context)):
                    context_vector[0][context[k]] = 1

                step1 = np.matmul(context_vector, self.W)
                step2 = np.matmul(step1, self.C)

                softmax = self.softmax(step2)

                target_vector = np.zeros((1, self.dictionary_size))
                target_vector[0][target] = 1

                loss += self.cross_entropy_loss(softmax, target_vector)
                gradient = softmax - target_vector
                for i in range(self.vector_size):
                    for j in range(self.dictionary_size):
                        self.C[i][j] -= learning_rate * context_vector[0][i] * gradient[0][j]
                for i in context:
                    self.W[i] -= learning_rate * np.dot(self.C, gradient[0])
            print('Epoch {} loss: {}'.format(epoch, loss))
        return self.W

3900
['казак', 'едва', 'вымысел', 'шолохов', 'ильичто', 'казак', 'что', 'там', 'тень', 'наводить', 'сибирской', 'губерния', 'такой', 'кореню', 'бывать', 'хвастливо', 'убеждать', 'бунчука', 'чикамасов', 'прадед', 'наш', 'кровь', 'полили', 'говорить', 'земля', 'григорий', 'мелехов', 'оттого', 'мочь', 'родить', 'наш', 'чернозем', 'это', 'весь', 'правда', 'поступить', 'царский', 'служба', 'xviii', 'век', 'активно', 'участвовать', 'весь', 'военный', 'кампания', 'казачество', 'проливать', 'свой', 'кровь', 'уже', 'свой', 'земля', 'чужой', 'далёкий', 'страна', 'военный', 'подвиг', 'были', 'плата', 'рассрочка', 'земельный', 'надеть', 'право', 'самим', 'выбирать', 'станичный', 'окружных', 'атаман', 'казачество', 'день', 'война', 'расплачивалось', 'лихва', 'тихий', 'время', 'трудиться', 'богатело', 'здесь', 'лёгкий', 'мочь', 'возникнуть', 'убеждение', 'что', 'счастие', 'дело', 'рука', 'человек', 'подвиг', 'обязанный', 'лихость', 'казак', 'богатство', 'его', 'трудолюбию', 'отсюда', 'яркий', 'индив

In [3]:
obj = Word2Vec(text_changed, 4, 500)
embedding = obj.train(text_changed, 10, 0.1)

Epoch 0 loss: 66970.37144396566
Epoch 1 loss: 40373.88868512087
Epoch 2 loss: 11014.02639140411
Epoch 3 loss: 1094.8741713146542
Epoch 4 loss: 131.70914860817376
Epoch 5 loss: 30.646059262710246
Epoch 6 loss: 19.16284069259654
Epoch 7 loss: 9.452300081591583
Epoch 8 loss: 8.160472972284039
Epoch 9 loss: 7.3609086468745355


In [ ]:
!pip install torch

   ---------------------------------------- 0.0/204.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/204.1 MB ? eta -:--:--
   ---------------------------------------- 2.4/204.1 MB 8.9 MB/s eta 0:00:23
   - -------------------------------------- 6.0/204.1 MB 12.7 MB/s eta 0:00:16
   - -------------------------------------- 9.2/204.1 MB 13.9 MB/s eta 0:00:15
   -- ------------------------------------- 11.0/204.1 MB 14.0 MB/s eta 0:00:14
   --- ------------------------------------ 16.5/204.1 MB 15.3 MB/s eta 0:00:13
   --- ------------------------------------ 19.9/204.1 MB 15.3 MB/s eta 0:00:13
   ---- ----------------------------------- 23.3/204.1 MB 15.7 MB/s eta 0:00:12
   ----- ---------------------------------- 27.3/204.1 MB 16.0 MB/s eta 0:00:12
   ------ --------------------------------- 30.7/204.1 MB 16.2 MB/s eta 0:00:11
   ------ --------------------------------- 34.1/204.1 MB 16.3 MB/s eta 0:00:11
   ------- -------------------------------- 37.7/204.1 MB 16.4


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import torch
import torch.nn as nn

class Net(nn.Module):
    def __init__(self, vector_size, dictionary_size):
        super(Net, self).__init__()
        self.vector_size = vector_size
        self.dictionary_size = dictionary_size
        self.pipe = nn.Sequential(
            nn.Linear(self.vector_size, self.vector_size),

            nn.Linear(self.vector_size, 500),
            nn.ReLU(),

            nn.Linear(500, self.dictionary_size))
    def forward(self, x):
        x = x.float()
        return self.pipe(x)

model = Net(obj.vector_size, obj.dictionary_size)

x_train = []
y_train = []
for x, y in obj.dictionary_context.items():
    x_vector = np.zeros(obj.dictionary_size)
    for k in range(len(x)):
        x_vector[x[k]] = 1
    x_vector = np.matmul(x_vector, embedding)
    x_train.append(x_vector)
    y_train.append(y)

x_train = torch.FloatTensor(np.array(x_train))
y_train = torch.LongTensor(np.array(y_train))
train_loader = torch.utils.data.DataLoader(list(zip(x_train, y_train)), batch_size=64, shuffle=True)

from tqdm.auto import tqdm

all_loss = []
all_perplexity = []
def train(model, loader, criterion, optimizer, num_epoch):
    for t in tqdm(range(num_epoch)):
        total_loss = 0.0
        total_samples = 0

        for x_batch, y_batch in loader:
            y_pred = model(x_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            total_loss += loss.item() * y_batch.size(0)
            total_samples += y_batch.size(0)

        epoch_loss = total_loss / total_samples
        epoch_perplexity = np.exp(epoch_loss)
        all_perplexity.append(epoch_perplexity)
        all_loss.append(epoch_loss)
        print('Epoch {} \t'.format(t), 'Loss: {}'.format(np.round(epoch_loss, 6)))
        print('Epoch {} \t'.format(t), 'Perplexity: {}'.format(np.round(epoch_perplexity, 6)))
    return model

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
train(model, train_loader, criterion, optimizer, 120)

print(obj.dictionary_words)

  0%|          | 0/120 [00:00<?, ?it/s]

Epoch 0 	 Loss: 7.935725
Epoch 0 	 Perplexity: 2795.385914
Epoch 1 	 Loss: 7.278055
Epoch 1 	 Perplexity: 1448.16813
Epoch 2 	 Loss: 6.980242
Epoch 2 	 Perplexity: 1075.178092
Epoch 3 	 Loss: 6.809443
Epoch 3 	 Perplexity: 906.365575
Epoch 4 	 Loss: 6.647838
Epoch 4 	 Perplexity: 771.115516
Epoch 5 	 Loss: 6.477987
Epoch 5 	 Perplexity: 650.6596
Epoch 6 	 Loss: 6.287392
Epoch 6 	 Perplexity: 537.748977
Epoch 7 	 Loss: 6.081068
Epoch 7 	 Perplexity: 437.496219
Epoch 8 	 Loss: 5.873913
Epoch 8 	 Perplexity: 355.638022
Epoch 9 	 Loss: 5.646514
Epoch 9 	 Perplexity: 283.302241
Epoch 10 	 Loss: 5.409035
Epoch 10 	 Perplexity: 223.415947
Epoch 11 	 Loss: 5.166105
Epoch 11 	 Perplexity: 175.231035
Epoch 12 	 Loss: 4.902731
Epoch 12 	 Perplexity: 134.657015
Epoch 13 	 Loss: 4.626482
Epoch 13 	 Perplexity: 102.154029
Epoch 14 	 Loss: 4.349039
Epoch 14 	 Perplexity: 77.404077
Epoch 15 	 Loss: 4.064205
Epoch 15 	 Perplexity: 58.218631
Epoch 16 	 Loss: 3.783358
Epoch 16 	 Perplexity: 43.963423
Epo

In [5]:
def string_processing(input_string):
    input_string = re.sub(r'\d+', '', input_string)
    input_string = input_string.translate(translator).split()
    input_string = [word for word in input_string if len(word) >= 3]
    input_string = [nlp(word)[0].lemma_ for word in input_string]
    input_string = input_string[-obj.L_size:]
    vec = np.zeros(obj.dictionary_size)
    for i in input_string:
        vec[obj.dictionary_words[i]] = 1
    vec = np.matmul(vec, embedding)
    return vec

str1 = 'решило исход с Колчаком и Деникиным'
str1 = string_processing(str1)
output = model(torch.tensor(str1))
print(obj.dictionary_indexes[list(output).index(max(list(output)))])

борьба


In [6]:

str2 = 'семь собственников одновременно взвесили'
str2 = string_processing(str2)
output = model(torch.tensor(str2))
print(obj.dictionary_indexes[list(output).index(max(list(output)))])


когда


In [7]:
str3 = 'казак издавна трудится за семью'
str3 = string_processing(str3)
output = model(torch.tensor(str3))
print(obj.dictionary_indexes[list(output).index(max(list(output)))])

дон


In [8]:
str4 = 'возглавлявшие прежде ... восстания против'
str4 = string_processing(str4)
output = model(torch.tensor(str4))
print(obj.dictionary_indexes[list(output).index(max(list(output)))])

народный


In [9]:
str5 = 'долгий год дал основание'
str5 = string_processing(str5)
output = model(torch.tensor(str5))
print(obj.dictionary_indexes[list(output).index(max(list(output)))])

казачество


In [10]:
str6 = 'банда героически защищала атамана'
str6 = string_processing(str6)
output = model(torch.tensor(str6))
print(obj.dictionary_indexes[list(output).index(max(list(output)))])

свой


In [11]:
str7 = 'тяжкими бывает отношения древних'
str7 = string_processing(str7)
output = model(torch.tensor(str7))
print(obj.dictionary_indexes[list(output).index(max(list(output)))])

такой


In [12]:
str8 = 'матрос на улице упрекал злодея'
str8 = string_processing(str8)
output = model(torch.tensor(str8))
print(obj.dictionary_indexes[list(output).index(max(list(output)))])

зарубить


In [13]:
str9 = 'носитель типичного сознания среднего'
str9 = string_processing(str9)
output = model(torch.tensor(str9))
print(obj.dictionary_indexes[list(output).index(max(list(output)))])

казачество


In [14]:
str10 = 'отношение массы к агитатору враждебно'
str10 = string_processing(str10)
output = model(torch.tensor(str10))
print(obj.dictionary_indexes[list(output).index(max(list(output)))])

век


In [15]:
str11 = 'отчаянных людей  решающий выбор'
str11 = string_processing(str11)
output = model(torch.tensor(str11))
print(obj.dictionary_indexes[list(output).index(max(list(output)))])

сделать


In [ ]:

str12 = 'сила Шолохова первый роман'
str12 = string_processing(str12)
output = model(torch.tensor(str12))
print(obj.dictionary_indexes[list(output).index(max(list(output)))])